<a href="https://colab.research.google.com/github/Eswa2020/urban-parking-prediction-models/blob/master/Dubai_Parking_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing libraries and loading dataset(survey responses)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils import resample
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load the combined dataset
file_path = '/content/Combined_Parking_Survey_Data.csv'
df = pd.read_csv(file_path)

In [ ]:
#Explore the dataset
df.shape

## Data Exploration

In [ ]:
#we check the columns list
#this also gives us a feel of how the question in the survey looked
df.columns.tolist()

In [ ]:
#we check for any missing values
df.isnull().sum()

In [ ]:
#we also check what are the datatypes of our data
df.dtypes

In [ ]:
# Display summary statistics for numerical columns
summary_stats = df.describe(include='all')
summary_stats

## Resampling -- using  Bayesian Bootrstrapping method

In [ ]:
from sklearn.utils import resample
import pandas as pd

# Original data size
original_size = len(df)
original_size

In [ ]:
# Desired total size (minimum 196)
target_size = 196

# Number of samples to add
n_to_add = target_size - original_size


In [ ]:
# Resample with replacement
resampled_df = resample(df, replace=True, n_samples=n_to_add, random_state=42)

# Concatenate to form the extended DataFrame
df = pd.concat([df, resampled_df], ignore_index=True)

# Confirm new size
print(df.shape)

## Data Pre-processing

In [ ]:
# List of columns to drop
columns_to_drop = [
    "_id",
    "_uuid",
    "_submission_time",
    "_validation_status",
    "_notes",
    "_status",
    "_submitted_by",
    "__version__",
    "_tags",
    "_index"
]

# Drop the columns
df = df.drop(columns=columns_to_drop)

In [ ]:
df.isnull().sum()

In [ ]:
# Clean all object (string) columns by stripping whitespace and replacing empty strings with NaN
df_cleaned = df.copy()
for col in df_cleaned.select_dtypes(include='object').columns:
    df_cleaned[col] = df_cleaned[col].astype(str).str.strip()
    df_cleaned[col] = df_cleaned[col].replace('', pd.NA)

In [ ]:
# Only drop columns that are completely empty (all values are NaN)
df = df.dropna(axis=1, how='all')

In [ ]:
# Check % of missing values per column
missing_percentage = df.isnull().mean().sort_values(ascending=False)
print(missing_percentage)

In [ ]:
threshold = 0.2
df = df.loc[:, df.isnull().mean() < threshold]

In [ ]:
# Check % of missing values per column
missing_percentage = df.isnull().mean().sort_values(ascending=False)
print(missing_percentage)

## Univariate Data analysis

In [ ]:
# Calculate value counts for labels
gender_counts = df["What is your gender?"].value_counts()
gender_counts

In [ ]:
# Separate categorical and numerical columns
categorical_cols = df.select_dtypes(include='object').columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
# Univariate analysis for categorical variables
print("📊 Categorical Variable Distributions")
for col in categorical_cols:
    value_counts = df[col].value_counts(dropna=False)
    if value_counts.empty:
        print(f"⏩ Skipping empty column: {col}")
        continue
    print(f"✅ Plotting: {col}")
    plt.figure(figsize=(10, 5))
    value_counts.plot(kind='bar')
    plt.title(f'Distribution of "{col}"')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.grid(axis='y')
    plt.show()

In [ ]:
# Univariate analysis for numerical variables
for col in numerical_cols:
    if df[col].dropna().empty:
        continue  # Skip if column has no numeric values
    plt.figure(figsize=(8, 4))
    df[col].dropna().hist(bins=15)
    plt.title(f'Distribution of "{col}"')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.grid(axis='y')
    plt.show()

## Bi-variate Analysis

In [ ]:
# Select only the relevant categorical columns for bivariate analysis
cat_cols = [
    "What is your age group?",
    "Where do you usually park your car?",
    "How often do you park in public paid parking?",
    "Do you adjust your daily routine to find parking?",
    "What is your top priority when choosing a parking spot?",
    "Choice Card: Option A vs Option B"
]

In [ ]:
# Clean dataset (drop rows with missing values in selected columns)
df_clean = df[cat_cols].dropna()

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Select only the relevant categorical columns for bivariate analysis
cat_cols = [
    "What is your age group?",
    "Where do you usually park your car?",
    "How often do you park in public paid parking?",
    "Do you adjust your daily routine to find parking?",
    "What is your top priority when choosing a parking spot?",
    "Choice Card: Option A vs Option B"
]

# Clean dataset (drop rows with missing values in selected columns)
df_clean = df[cat_cols].dropna()

# Perform chi-square test and create cross-tabs with heatmaps
chi_square_results = {}

for i in range(len(cat_cols)):
    for j in range(i + 1, len(cat_cols)):
        var1 = cat_cols[i]
        var2 = cat_cols[j]
        contingency = pd.crosstab(df_clean[var1], df_clean[var2])
        chi2, p, dof, expected = chi2_contingency(contingency)
        chi_square_results[f"{var1} vs {var2}"] = {"Chi2": chi2, "p-value": p, "dof": dof}

chi_square_results

In [ ]:
# Heatmap for "What is your age group?" vs "Where do you usually park your car?"
var1 = "What is your age group?"
var2 = "Where do you usually park your car?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(14, 8))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "What is your age group?" vs "How often do you park in public paid parking?"
var1 = "What is your age group?"
var2 = "How often do you park in public paid parking?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "What is your age group?" vs "Do you adjust your daily routine to find parking?"
var1 = "What is your age group?"
var2 = "Do you adjust your daily routine to find parking?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "What is your age group?" vs "What is your top priority when choosing a parking spot?"
var1 = "What is your age group?"
var2 = "What is your top priority when choosing a parking spot?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "What is your age group?" vs "Choice Card: Option A vs Option B"
var1 = "What is your age group?"
var2 = "Choice Card: Option A vs Option B"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "Where do you usually park your car?" vs "How often do you park in public paid parking?"
var1 = "Where do you usually park your car?"
var2 = "How often do you park in public paid parking?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "Where do you usually park your car?" vs "Do you adjust your daily routine to find parking?"
var1 = "Where do you usually park your car?"
var2 = "Do you adjust your daily routine to find parking?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "Where do you usually park your car?" vs "What is your top priority when choosing a parking spot?"
var1 = "Where do you usually park your car?"
var2 = "What is your top priority when choosing a parking spot?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "Where do you usually park your car?" vs "Choice Card: Option A vs Option B"
var1 = "Where do you usually park your car?"
var2 = "Choice Card: Option A vs Option B"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "How often do you park in public paid parking?" vs "Do you adjust your daily routine to find parking?"
var1 = "How often do you park in public paid parking?"
var2 = "Do you adjust your daily routine to find parking?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "How often do you park in public paid parking?" vs "What is your top priority when choosing a parking spot?"
var1 = "How often do you park in public paid parking?"
var2 = "What is your top priority when choosing a parking spot?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "How often do you park in public paid parking?" vs "Choice Card: Option A vs Option B"
var1 = "How often do you park in public paid parking?"
var2 = "Choice Card: Option A vs Option B"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "Do you adjust your daily routine to find parking?" vs "What is your top priority when choosing a parking spot?"
var1 = "Do you adjust your daily routine to find parking?"
var2 = "What is your top priority when choosing a parking spot?"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "Do you adjust your daily routine to find parking?" vs "Choice Card: Option A vs Option B"
var1 = "Do you adjust your daily routine to find parking?"
var2 = "Choice Card: Option A vs Option B"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for "What is your top priority when choosing a parking spot?" vs "Choice Card: Option A vs Option B"
var1 = "What is your top priority when choosing a parking spot?"
var2 = "Choice Card: Option A vs Option B"
contingency = pd.crosstab(df_clean[var1], df_clean[var2])
plt.figure(figsize=(10, 6))
sns.heatmap(contingency, annot=True, fmt='d', cmap="Blues")
plt.title(f"Heatmap: {var1} vs {var2}")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Redefine variable pairs
age_vs_priority = df[['What is your age group?', 'What is your top priority when choosing a parking spot?']].dropna()


In [ ]:
# Plot 1: Grouped bar plot for Age Group vs Parking Priority
plt.figure(figsize=(12, 6))
sns.countplot(data=age_vs_priority, x='What is your age group?', hue='What is your top priority when choosing a parking spot?')
plt.title('Parking Priority by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.legend(title='Top Parking Priority', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



In [ ]:
location_vs_routine = df[['Where do you usually park your car?', 'Do you adjust your daily routine to find parking?']].dropna()

In [ ]:
# Create the location_vs_routine DataFrame
location_vs_routine = df[['Where do you usually park your car?', 'Do you adjust your daily routine to find parking?']]

# Plot 2: Grouped bar plot for Location vs Routine Adjustment
plt.figure(figsize=(12, 6))
sns.countplot(data=location_vs_routine, x='Where do you usually park your car?', hue='Do you adjust your daily routine to find parking?')
plt.title('Routine Adjustment by Usual Parking Location')
plt.xlabel('Usual Parking Location')
plt.ylabel('Count')
plt.legend(title='Adjust Daily Routine', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Multi-variate analysis

In [ ]:
# Encode categorical variables into numerical form for correlation
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = df_encoded[col].astype('category').cat.codes

# Calculate the correlation matrix
correlation_matrix = df_encoded.corr()


In [ ]:
# Plot the correlation matrix using a heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, linewidths=.5)
plt.title("Correlation Matrix of Encoded Parking Survey Data")
plt.tight_layout()
plt.show()

Strong correlations indicate that two variables may be capturing similar trends (e.g., Distance vs Price).

Weak or near-zero correlations imply independence.

If you observe high correlations (e.g., > 0.7 or < -0.7), especially among input features, you may want to address multicollinearity (e.g., via PCA or feature selection).

In [ ]:
from sklearn.impute import SimpleImputer

# Initialize imputer with strategy (mean, median, most_frequent, constant)
imputer = SimpleImputer(strategy='mean')

# Fit and transform the data to replace NaNs
df_imputed = imputer.fit_transform(df_encoded)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the imputed data
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_imputed)

# Perform PCA
pca = PCA(n_components=5)
pca_result = pca.fit(df_scaled)

In [ ]:
# Scree plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_, marker='o')
plt.title('Scree Plot: Explained Variance by Principal Components')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.grid(True)
plt.xticks(range(1, len(pca.explained_variance_ratio_) + 1))
plt.tight_layout()
plt.show()

In [ ]:
# Identify high correlations (> 0.6 or < -0.6)
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > 0.6:
            high_corr_pairs.append((correlation_matrix.columns[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j]))

In [ ]:
# Display the high correlation pairs
display(high_corr_pairs)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF
vif_data = pd.DataFrame()
vif_data['Feature'] = df_encoded.columns
vif_data['VIF'] = [variance_inflation_factor(df_scaled, i) for i in range(df_scaled.shape[1])]

In [ ]:
# Display top VIF values
print(vif_data.sort_values(by="VIF", ascending=False))

In [ ]:
display(vif_data)

## Co-Joint Analysis

In [ ]:
import numpy as np
import re
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Step 1: Extract relevant choice card data
df_choice = df[['Choice Card: Option A vs Option B']].dropna()

In [ ]:
# Step 2: Parse and encode choice card attributes (e.g., price, shade, distance, security)
def extract_attributes(row):
    text = row['Choice Card: Option A vs Option B']
    match = re.search(r'\((\d+) AED/hr, (Yes|No) shade, (\d+) min walk, (Easy|Hard)\)', text)
    if match:
        price = int(match.group(1))
        shade = match.group(2)
        distance = int(match.group(3))
        difficulty = match.group(4)
        return pd.Series([price, shade, distance, difficulty])
    return pd.Series([np.nan, np.nan, np.nan, np.nan])

df_choice[['Price', 'Shade', 'Distance', 'Security']] = df_choice.apply(extract_attributes, axis=1)

In [ ]:
# Step 3: Clean and encode
df_choice.dropna(inplace=True)
df_choice['Choice'] = df_choice['Choice Card: Option A vs Option B'].apply(lambda x: 1 if 'Option B' in x else 0)
X = df_choice[['Price', 'Shade', 'Distance', 'Security']]
y = df_choice['Choice']

In [ ]:
# One-hot encode categorical variables
X_encoded = pd.get_dummies(X, columns=['Shade', 'Security'], drop_first=True)

In [ ]:
# Step 4: Logistic Regression for Conjoint
log_reg = LogisticRegression()
log_reg.fit(X_encoded, y)

In [ ]:
# Step 5: Visualize Coefficients
coefficients = pd.Series(log_reg.coef_[0], index=X_encoded.columns)
plt.figure(figsize=(10, 6))
sns.barplot(x=coefficients.values, y=coefficients.index)
plt.title("Part-Worth Utilities from Conjoint Analysis")
plt.xlabel("Coefficient Value (Impact on Preference)")
plt.axvline(0, color='grey', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Also return coefficients for explanation
coefficients.sort_values(ascending=False)

## Simulate MarKet Segments

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Scale data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_imputed)  # Use your cleaned + encoded data

# Choose number of clusters (try 2–5)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_encoded['Segment'] = kmeans.fit_predict(scaled_data)

# Visualize cluster sizes
df_encoded['Segment'].value_counts().plot(kind='bar', title='User Segments')
plt.xlabel("Segment")
plt.ylabel("Number of Users")
plt.show()

In [ ]:
import numpy as np

# Feature order used during model training (update if different):
# [Price, Distance, Shade, Security]

# Example input: you want to simulate a user choosing a parking option with:
# Price = 2 AED, Distance = 100m, Shade = 1 (Yes), Security = 1 (Yes)

features = np.array([[2, 100, 1, 1]])  # shape = (1, 4)

# Make sure feature names match what the model was trained on
prediction = log_reg.predict_proba(features)

# Output probability of choosing this option
print("Probability of choosing this option:", prediction[0][1])


In [ ]:
import numpy as np

# Example feature input: price = 2 AED, distance = 100m, Shade = No, Security = Yes
# Ensure the features are in the same order and format as the training data (X_encoded)
# X_encoded columns: 'Price', 'Distance', 'Shade_Yes', 'Security_No'
features = np.array([[2, 100, False, False]]) # Assuming Shade is 'No' and Security is 'Yes'

prediction = log_reg.predict_proba(features)
print("Probability of choosing this option:", prediction[0][1])